# 🏏 IPL Match Win Predictor (Logistic Regression)

This project aims to predict the outcome of IPL (Indian Premier League) matches (Second Inning Prediction) using machine learning techniques on historical data.

Two CSV files are used:
- `matches.csv`: Contains high-level match information (teams, venue, winner, toss, etc.)
- `deliveries.csv`: Contains ball-by-ball delivery data for each match

## Project Workflow

## 1. Data Import & Exploration
- Imported necessary libraries: `pandas`, `numpy`, `sklearn`, `matplotlib`.
- Loaded datasets using `pd.read_csv`.
- Displayed top rows and basic info of both datasets to understand the structure.

In [1]:
# Importing libraries

import numpy as np
import pandas as pd


In [2]:
# importing data

match = pd.read_csv('matches.csv')
delivery = pd.read_csv('deliveries.csv')

A Glance at match datset

In [3]:
match.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [4]:
print(match.shape)
match.tail()

(1095, 20)


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
1090,1426307,2024,Hyderabad,2024-05-19,League,Abhishek Sharma,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Punjab Kings,Sunrisers Hyderabad,Punjab Kings,bat,Sunrisers Hyderabad,wickets,4.0,215.0,20.0,N,NaN,Nitin Menon,VK Sharma
1091,1426309,2024,Ahmedabad,2024-05-21,Qualifier 1,MA Starc,"Narendra Modi Stadium, Ahmedabad",Sunrisers Hyderabad,Kolkata Knight Riders,Sunrisers Hyderabad,bat,Kolkata Knight Riders,wickets,8.0,160.0,20.0,N,NaN,AK Chaudhary,R Pandit
1092,1426310,2024,Ahmedabad,2024-05-22,Eliminator,R Ashwin,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,Rajasthan Royals,Rajasthan Royals,field,Rajasthan Royals,wickets,4.0,173.0,20.0,N,NaN,KN Ananthapadmanabhan,MV Saidharshan Kumar
1093,1426311,2024,Chennai,2024-05-24,Qualifier 2,Shahbaz Ahmed,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,Rajasthan Royals,Rajasthan Royals,field,Sunrisers Hyderabad,runs,36.0,176.0,20.0,N,NaN,Nitin Menon,VK Sharma
1094,1426312,2024,Chennai,2024-05-26,Final,MA Starc,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,Kolkata Knight Riders,Sunrisers Hyderabad,bat,Kolkata Knight Riders,wickets,8.0,114.0,20.0,N,NaN,J Madanagopal,Nitin Menon


A Glance at delivery dataset

In [5]:
delivery.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [6]:
delivery.shape

(260920, 17)

## 2. Data Cleaning & Preprocessing
- Removed null values and matches with no results or duplicates.
- Standardized team names for consistency.
- Filtered relevant columns from both datasets.
- Merged match data with delivery data to form a unified dataset.

Calculating total score after the completion of an innning

In [7]:
total_score_df = delivery.groupby(['match_id','inning']).sum()['total_runs'].reset_index()

total_score_df

,match_id,inning,total_runs
0,335982,1,222
1,335982,2,82
2,335983,1,240
3,335983,2,207
4,335984,1,129
...,...,...,...
2212,1426310,2,174
2213,1426311,1,175
2214,1426311,2,139
2215,1426312,1,113


Calculating total score for first innning

In [8]:
total_score_df = total_score_df[total_score_df['inning'] == 1]

In [9]:
total_score_df

,match_id,inning,total_runs
0,335982,1,222
2,335983,1,240
4,335984,1,129
6,335985,1,165
8,335986,1,110
...,...,...,...
2207,1426307,1,214
2209,1426309,1,159
2211,1426310,1,172
2213,1426311,1,175


Combining the match dataframe to the total _score_df for the runs scored in the first inning

In [10]:
match_df = match.merge(total_score_df[['match_id','total_runs']],left_on='id',right_on='match_id')

In [11]:
match_df.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,match_id,total_runs
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,335982,222
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,...,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri,335983,240
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,...,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar,335984,129
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,...,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper,335985,165
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,...,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan,335986,110


### Data Preprocessing


Checking the teams that has played IPL and eliminating those who donot play anymore.

In [12]:
match_df['team1'].unique()

<ArrowStringArray>
['Royal Challengers Bangalore',             'Kings XI Punjab',
            'Delhi Daredevils',              'Mumbai Indians',
       'Kolkata Knight Riders',            'Rajasthan Royals',
             'Deccan Chargers',         'Chennai Super Kings',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',               'Gujarat Lions',
     'Rising Pune Supergiants',      'Rising Pune Supergiant',
              'Delhi Capitals',                'Punjab Kings',
        'Lucknow Super Giants',              'Gujarat Titans',
 'Royal Challengers Bengaluru']
Length: 19, dtype: str

In [13]:
# Teams playing the present IPL season

teams = [
    'Sunrisers Hyderabad',
    'Mumbai Indians',
    'Royal Challengers Bengaluru',
    'Kolkata Knight Riders',
    'Punjab Kings',
    'Chennai Super Kings',
    'Rajasthan Royals',
    'Delhi Capitals',
    'Gujarat Titans',
    'Lucknow Super Giants'
]

In [14]:
# Changing previous names of team with present names in match_df dataframe

match_df['team1'] = match_df['team1'].str.replace('Delhi Daredevils','Delhi Capitals')
match_df['team2'] = match_df['team2'].str.replace('Delhi Daredevils','Delhi Capitals')
match_df['winner'] = match_df['winner'].str.replace('Delhi Daredevils','Delhi Capitals')


match_df['team1'] = match_df['team1'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
match_df['team2'] = match_df['team2'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
match_df['winner'] = match_df['winner'].str.replace('Deccan Chargers','Sunrisers Hyderabad')


match_df['team1'] = match_df['team1'].str.replace('Royal Challengers Bangalore', 'Royal Challengers Bengaluru')
match_df['team2'] = match_df['team2'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')
match_df['winner'] = match_df['winner'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')


match_df['team1'] = match_df['team1'].str.replace('Kings XI Punjab','Punjab Kings')
match_df['team2'] = match_df['team2'].str.replace('Kings XI Punjab','Punjab Kings')
match_df['winner'] = match_df['winner'].str.replace('Kings XI Punjab','Punjab Kings')


match_df['city'] = match_df['city'].str.replace('Bangalore','Bengaluru')
match_df['team1'].unique()

<ArrowStringArray>
['Royal Challengers Bengaluru',                'Punjab Kings',
              'Delhi Capitals',              'Mumbai Indians',
       'Kolkata Knight Riders',            'Rajasthan Royals',
         'Sunrisers Hyderabad',         'Chennai Super Kings',
        'Kochi Tuskers Kerala',               'Pune Warriors',
               'Gujarat Lions',     'Rising Pune Supergiants',
      'Rising Pune Supergiant',        'Lucknow Super Giants',
              'Gujarat Titans']
Length: 15, dtype: str

In [15]:
match_df

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,match_id,total_runs
0,335982,2007/08,Bengaluru,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bangalore,...,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,335982,222
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,...,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri,335983,240
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,...,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar,335984,129
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,...,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper,335985,165
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Sunrisers Hyderabad,Deccan Chargers,...,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan,335986,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,1426307,2024,Hyderabad,2024-05-19,League,Abhishek Sharma,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Punjab Kings,Sunrisers Hyderabad,Punjab Kings,...,wickets,4.0,215.0,20.0,N,NaN,Nitin Menon,VK Sharma,1426307,214
1091,1426309,2024,Ahmedabad,2024-05-21,Qualifier 1,MA Starc,"Narendra Modi Stadium, Ahmedabad",Sunrisers Hyderabad,Kolkata Knight Riders,Sunrisers Hyderabad,...,wickets,8.0,160.0,20.0,N,NaN,AK Chaudhary,R Pandit,1426309,159
1092,1426310,2024,Ahmedabad,2024-05-22,Eliminator,R Ashwin,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,Rajasthan Royals,Rajasthan Royals,...,wickets,4.0,173.0,20.0,N,NaN,KN Ananthapadmanabhan,MV Saidharshan Kumar,1426310,172
1093,1426311,2024,Chennai,2024-05-24,Qualifier 2,Shahbaz Ahmed,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,Rajasthan Royals,Rajasthan Royals,...,runs,36.0,176.0,20.0,N,NaN,Nitin Menon,VK Sharma,1426311,175


In [16]:
delivery['batting_team'].unique()

<ArrowStringArray>
[      'Kolkata Knight Riders', 'Royal Challengers Bangalore',
         'Chennai Super Kings',             'Kings XI Punjab',
            'Rajasthan Royals',            'Delhi Daredevils',
              'Mumbai Indians',             'Deccan Chargers',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',     'Rising Pune Supergiants',
               'Gujarat Lions',      'Rising Pune Supergiant',
              'Delhi Capitals',                'Punjab Kings',
        'Lucknow Super Giants',              'Gujarat Titans',
 'Royal Challengers Bengaluru']
Length: 19, dtype: str

In [17]:
# Changing previous names of team with present names in delivery dataframe

delivery['batting_team'] = delivery['batting_team'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')
delivery['bowling_team'] = delivery['bowling_team'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')

delivery['batting_team'] = delivery['batting_team'].str.replace('Delhi Daredevils','Delhi Capitals')
delivery['bowling_team'] = delivery['bowling_team'].str.replace('Delhi Daredevils','Delhi Capitals')

delivery['batting_team'] = delivery['batting_team'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
delivery['bowling_team'] = delivery['bowling_team'].str.replace('Deccan Chargers','Sunrisers Hyderabad')


delivery['batting_team'] = delivery['batting_team'].str.replace('Kings XI Punjab','Punjab Kings')
delivery['bowling_team'] = delivery['bowling_team'].str.replace('Kings XI Punjab','Punjab Kings')

delivery['batting_team'].unique()

<ArrowStringArray>
[      'Kolkata Knight Riders', 'Royal Challengers Bengaluru',
         'Chennai Super Kings',                'Punjab Kings',
            'Rajasthan Royals',              'Delhi Capitals',
              'Mumbai Indians',         'Sunrisers Hyderabad',
        'Kochi Tuskers Kerala',               'Pune Warriors',
     'Rising Pune Supergiants',               'Gujarat Lions',
      'Rising Pune Supergiant',        'Lucknow Super Giants',
              'Gujarat Titans']
Length: 15, dtype: str

In [18]:
# This checks if the team listed in the team1, team2 columns are present in the list teams.

match_df = match_df[match_df['team1'].isin(teams)]
match_df = match_df[match_df['team2'].isin(teams)]

In [19]:
match_df.shape

(980, 22)

In [20]:
match_df = match_df[['match_id','city','winner','total_runs']]
match_df

,match_id,city,winner,total_runs
0,335982,Bengaluru,Kolkata Knight Riders,222
1,335983,Chandigarh,Chennai Super Kings,240
2,335984,Delhi,Delhi Capitals,129
3,335985,Mumbai,Royal Challengers Bengaluru,165
4,335986,Kolkata,Kolkata Knight Riders,110
...,...,...,...,...
1090,1426307,Hyderabad,Sunrisers Hyderabad,214
1091,1426309,Ahmedabad,Kolkata Knight Riders,159
1092,1426310,Ahmedabad,Rajasthan Royals,172
1093,1426311,Chennai,Sunrisers Hyderabad,175


In [21]:
# Merging match_df(with required/relevent features) with delivery_df using match_id as key 

delivery_df = match_df.merge(delivery,on='match_id')

In [22]:
delivery_df = delivery_df[delivery_df['inning'] == 2]
delivery_df 

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs_y,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
124,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,1,R Dravid,AB Dinda,W Jaffer,1,0,1,NaN,0,NaN,NaN,NaN
125,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,2,W Jaffer,AB Dinda,R Dravid,0,1,1,wides,0,NaN,NaN,NaN
126,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,3,W Jaffer,AB Dinda,R Dravid,0,0,0,NaN,0,NaN,NaN,NaN
127,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,4,W Jaffer,AB Dinda,R Dravid,1,0,1,NaN,0,NaN,NaN,NaN
128,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,5,R Dravid,AB Dinda,W Jaffer,1,0,1,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233896,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,5,SS Iyer,AK Markram,VR Iyer,1,0,1,NaN,0,NaN,NaN,NaN
233897,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,6,VR Iyer,AK Markram,SS Iyer,1,0,1,NaN,0,NaN,NaN,NaN
233898,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,1,VR Iyer,Shahbaz Ahmed,SS Iyer,1,0,1,NaN,0,NaN,NaN,NaN
233899,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,2,SS Iyer,Shahbaz Ahmed,VR Iyer,1,0,1,NaN,0,NaN,NaN,NaN


## 3. Feature Engineering

3. Feature Engineering
Engineered key features for the prediction model:
- **Current Score**: Sum of runs scored till that ball.
- **Runs Left**: Target - Current Score.
- **Balls Left**: Based on overs completed.
- **Wickets Left**: 10 - number of wickets fallen.
- **Current Run Rate**: Current score / overs faced.
- **Required Run Rate**: Runs left / balls left.
- **Result**: Whether batting team won the match.

In [23]:
# calculate a running total of total_runs for each ball (i.e., cumulative score up to that ball).

delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs_y'].cumsum()

In [24]:
delivery_df['runs_left'] = (delivery_df['total_runs_x'] + 1) - delivery_df['current_score']

In [25]:
delivery_df['balls_left'] = 120 - (delivery_df['over']*6 + delivery_df['ball'])

In [26]:
wickets = delivery_df['wickets_left'] = delivery_df.groupby('match_id')['is_wicket'].cumsum()

delivery_df['wickets_left'] = 10 - wickets
delivery_df.head(50)

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batter,...,total_runs_y,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets_left
124,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,1,R Dravid,...,1,NaN,0,NaN,NaN,NaN,1,222,119,10
125,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,2,W Jaffer,...,1,wides,0,NaN,NaN,NaN,2,221,118,10
126,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,3,W Jaffer,...,0,NaN,0,NaN,NaN,NaN,2,221,117,10
127,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,4,W Jaffer,...,1,NaN,0,NaN,NaN,NaN,3,220,116,10
128,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,5,R Dravid,...,1,NaN,0,NaN,NaN,NaN,4,219,115,10
129,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,6,W Jaffer,...,0,NaN,0,NaN,NaN,NaN,4,219,114,10
130,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,7,W Jaffer,...,0,NaN,0,NaN,NaN,NaN,4,219,113,10
131,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,1,1,R Dravid,...,0,NaN,1,R Dravid,bowled,NaN,4,219,113,9
132,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,1,2,V Kohli,...,0,NaN,0,NaN,NaN,NaN,4,219,112,9
133,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,1,3,V Kohli,...,4,legbyes,0,NaN,NaN,NaN,8,215,111,9


In [27]:
# crr = runs/overs

delivery_df['crr'] = (delivery_df['current_score']*6)/(120 - delivery_df['balls_left'])

np.isinf(delivery_df['crr']).sum()

np.int64(0)

In [28]:
# rrr = (runs required)/(overs left)

delivery_df['rrr'] = (delivery_df['runs_left']*6)/delivery_df['balls_left']

np.isinf(delivery_df['rrr']).sum()

np.int64(372)

In [29]:
delivery_df.tail(70)

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batter,...,is_wicket,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets_left,crr,rrr
233714,1426311,Chennai,Sunrisers Hyderabad,175,2,Rajasthan Royals,Sunrisers Hyderabad,19,4,Dhruv Jurel,...,0,NaN,NaN,NaN,138,38,2,3,7.016949,114.000000
233715,1426311,Chennai,Sunrisers Hyderabad,175,2,Rajasthan Royals,Sunrisers Hyderabad,19,5,Dhruv Jurel,...,0,NaN,NaN,NaN,138,38,1,3,6.957983,228.000000
233716,1426311,Chennai,Sunrisers Hyderabad,175,2,Rajasthan Royals,Sunrisers Hyderabad,19,6,Dhruv Jurel,...,0,NaN,NaN,NaN,139,37,0,3,6.950000,inf
233834,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,0,1,Rahmanullah Gurbaz,...,0,NaN,NaN,NaN,0,114,119,10,0.000000,5.747899
233835,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,0,2,Rahmanullah Gurbaz,...,0,NaN,NaN,NaN,4,110,118,10,12.000000,5.593220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233896,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,5,SS Iyer,...,0,NaN,NaN,NaN,110,4,61,8,11.186441,0.393443
233897,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,6,VR Iyer,...,0,NaN,NaN,NaN,111,3,60,8,11.100000,0.300000
233898,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,1,VR Iyer,...,0,NaN,NaN,NaN,112,2,59,8,11.016393,0.203390
233899,1426312,Chennai,Kolkata Knight Riders,113,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,2,SS Iyer,...,0,NaN,NaN,NaN,113,1,58,8,10.935484,0.103448


In [30]:
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

In [31]:
delivery_df['result'] = delivery_df.apply(result,axis=1)
delivery_df.head()

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batter,...,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets_left,crr,rrr,result
124,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,1,R Dravid,...,NaN,NaN,NaN,1,222,119,10,6.0,11.193277,0
125,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,2,W Jaffer,...,NaN,NaN,NaN,2,221,118,10,6.0,11.237288,0
126,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,3,W Jaffer,...,NaN,NaN,NaN,2,221,117,10,4.0,11.333333,0
127,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,4,W Jaffer,...,NaN,NaN,NaN,3,220,116,10,4.5,11.379310,0
128,335982,Bengaluru,Kolkata Knight Riders,222,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,5,R Dravid,...,NaN,NaN,NaN,4,219,115,10,4.8,11.426087,0


Final dataset for model training

In [32]:
final_df = delivery_df[['batting_team','bowling_team','city','runs_left','balls_left','wickets_left','total_runs_x','crr','rrr','result']]

In [33]:
# Shuffling the data

final_df = final_df.sample(final_df.shape[0])

In [34]:
final_df.sample()

,batting_team,bowling_team,city,runs_left,balls_left,wickets_left,total_runs_x,crr,rrr,result
186032,Sunrisers Hyderabad,Gujarat Titans,Mumbai,122,85,10,162,7.028571,8.611765,1


In [35]:
final_df.dropna(inplace=True)

In [36]:
final_df.isnull().sum()

batting_team    0
bowling_team    0
city            0
runs_left       0
balls_left      0
wickets_left    0
total_runs_x    0
crr             0
rrr             0
result          0
dtype: int64

In [37]:
final_df = final_df[final_df['balls_left'] != 0]

## 4. Model Preparation
- Categorical features (team names, venue) were encoded.
- Split the dataset into:
  - `X`: Feature set
  - `y`: Target label (`1` for win, `0` for loss)

In [38]:
X = final_df.iloc[:,:-1]
y = final_df.iloc[:,-1]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

ModuleNotFoundError: No module named 'sklearn'

In [39]:
X_train

NameError: name 'X_train' is not defined

Appling one hot encoder for categorical variables

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

trf = ColumnTransformer([
    ('trf',OneHotEncoder(sparse_output=False,drop='first'),['batting_team','bowling_team','city'])
]
,remainder='passthrough')

X_train

,batting_team,bowling_team,city,runs_left,balls_left,wickets_left,total_runs_x,crr,rrr
105846,Rajasthan Royals,Chennai Super Kings,Chennai,67,33,5,157,6.275862,12.181818
39377,Sunrisers Hyderabad,Punjab Kings,Dharamsala,116,77,8,174,8.232558,9.038961
143983,Delhi Capitals,Kolkata Knight Riders,Kolkata,90,59,8,178,8.754098,9.152542
8161,Delhi Capitals,Kolkata Knight Riders,Kolkata,107,97,8,133,7.043478,6.618557
16058,Delhi Capitals,Royal Challengers Bengaluru,Port Elizabeth,139,109,9,149,6.000000,7.651376
...,...,...,...,...,...,...,...,...,...
176855,Royal Challengers Bengaluru,Rajasthan Royals,Dubai,142,115,10,149,9.600000,7.408696
30609,Delhi Capitals,Sunrisers Hyderabad,Cuttack,137,96,9,171,8.750000,8.562500
51947,Mumbai Indians,Kolkata Knight Riders,Mumbai,62,62,8,147,8.896552,6.000000
43405,Royal Challengers Bengaluru,Sunrisers Hyderabad,Hyderabad,150,92,8,175,5.571429,9.782609


## 5. Model Training
- Trained a **Logistic Regression** using Scikit-learn.
- Used `train_test_split()` for validation.
- Tuned model parameters for better accuracy.

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [42]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',LogisticRegression(solver='liblinear'))
])

In [43]:
pipe.fit(X_train,y_train)

C:\Users\pawan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('step1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('trf',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['batting_team',
                                                   'bowling_team', 'city'])])),
                ('step2', LogisticRegression(solver='liblinear'))])

In [44]:
y_pred = pipe.predict(X_test)

## 6. Evaluation
- Evaluated model accuracy on test data.
- Confusion matrix and classification report were used for performance metrics.

In [45]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.800338250493282

In [46]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))
